In [ ]:
# Import libraries
import json
import os
from pyspark.sql import SparkSession, functions as F # Spark Session and functions
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType # Create schema
from pyspark.sql import Window as W # Add window query details

In [2]:
# Verify parent folder, store confirguration file's path
folder = "./misc/"
os.makedirs(folder, exist_ok=True)
filepath = os.path.join(folder, 'config_pyspark.txt')

# Open config file, store file's content (environment varibles dictionary) as dictionary
with open(filepath) as configs:
    spark_configs = json.load(configs)
    
# Set environment variables
for k, v in spark_configs.items(): os.environ[k] = v

In [3]:
# Create schema object of cleaned data
crime_schema = StructType([
    StructField(name="DR_NO", dataType=IntegerType(), nullable=True),
    StructField(name="DATE_RPTD", dataType=StringType(), nullable=True),
    StructField(name="DATE_OCC", dataType=StringType(), nullable=True),
    StructField(name="TIME_OCC", dataType=IntegerType(), nullable=True),
    StructField(name="AREA", dataType=IntegerType(), nullable=True),
    StructField(name="AREA_NAME", dataType=StringType(), nullable=True),
    StructField(name="RPT_DIST_NO", dataType=IntegerType(), nullable=True),
    StructField(name="PART_1_2", dataType=IntegerType(), nullable=True),
    StructField(name="CRM_CD", dataType=IntegerType(), nullable=True),
    StructField(name="CRM_CD_DESC", dataType=StringType(), nullable=True),
    StructField(name="MOCODES", dataType=StringType(), nullable=True),
    StructField(name="VICT_AGE", dataType=IntegerType(), nullable=True),
    StructField(name="VICT_SEX", dataType=StringType(), nullable=True),
    StructField(name="PREMIS_CD", dataType=IntegerType(), nullable=True),
    StructField(name="PREMIS_DESC", dataType=StringType(), nullable=True),
    StructField(name="WEAPON_DESC", dataType=StringType(), nullable=True),
    StructField(name="STATUS_DESC", dataType=StringType(), nullable=True),
    StructField(name="CRM_CD_1", dataType=IntegerType(), nullable=True),
    StructField(name="CRM_CD_2", dataType=IntegerType(), nullable=True),
    StructField(name="CRM_CD_3", dataType=IntegerType(), nullable=True),
    StructField(name="CRM_CD_4", dataType=IntegerType(), nullable=True),
    StructField(name="LOCATION", dataType=StringType(), nullable=True),
    StructField(name="CROSS_STREET", dataType=StringType(), nullable=True),
    StructField(name="LAT", dataType=FloatType(), nullable=True),
    StructField(name="LON", dataType=FloatType(), nullable=True)
])

In [ ]:
# Create spark session
spark = SparkSession.builder.appName("explore").getOrCreate()

# Read data with new schema, print schema
crime_data_path = "./data/data-clean-pandas.csv"
crime = spark.read.csv(path=crime_data_path, 
                       inferSchema=False, 
                       schema=crime_schema, 
                       header=True)
crime.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/23 23:45:34 WARN Utils: Your hostname, Rob-Laptop, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/10/23 23:45:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/23 23:45:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- DR_NO: integer (nullable = true)
 |-- DATE_RPTD: string (nullable = true)
 |-- DATE_OCC: string (nullable = true)
 |-- TIME_OCC: integer (nullable = true)
 |-- AREA: integer (nullable = true)
 |-- AREA_NAME: string (nullable = true)
 |-- RPT_DIST_NO: integer (nullable = true)
 |-- PART_1_2: integer (nullable = true)
 |-- CRM_CD: integer (nullable = true)
 |-- CRM_CD_DESC: string (nullable = true)
 |-- MOCODES: string (nullable = true)
 |-- VICT_AGE: integer (nullable = true)
 |-- VICT_SEX: string (nullable = true)
 |-- PREMIS_CD: integer (nullable = true)
 |-- PREMIS_DESC: string (nullable = true)
 |-- WEAPON_DESC: string (nullable = true)
 |-- STATUS_DESC: string (nullable = true)
 |-- CRM_CD_1: integer (nullable = true)
 |-- CRM_CD_2: integer (nullable = true)
 |-- CRM_CD_3: integer (nullable = true)
 |-- CRM_CD_4: integer (nullable = true)
 |-- LOCATION: string (nullable = true)
 |-- CROSS_STREET: string (nullable = true)
 |-- LAT: float (nullable = true)
 |-- LON: float (

In [ ]:
# Show first 5 records
crime.head(5)

[Row(DR_NO=190326475, DATE_RPTD='03/01/2020 12:00:00 AM', DATE_OCC='03/01/2020 12:00:00 AM', TIME_OCC=2130, AREA=7, AREA_NAME='WILSHIRE', RPT_DIST_NO=784, PART_1_2=1, CRM_CD=510, CRM_CD_DESC='VEHICLE - STOLEN', MOCODES=None, VICT_AGE=0, VICT_SEX='M', PREMIS_CD=101, PREMIS_DESC='STREET', WEAPON_DESC=None, STATUS_DESC='ADULT ARREST', CRM_CD_1=510, CRM_CD_2=998, CRM_CD_3=0, CRM_CD_4=0, LOCATION='1900 S  LONGWOOD                     AV', CROSS_STREET=None, LAT=34.037498474121094, LON=-118.35060119628906),
 Row(DR_NO=200106753, DATE_RPTD='02/09/2020 12:00:00 AM', DATE_OCC='02/08/2020 12:00:00 AM', TIME_OCC=1800, AREA=1, AREA_NAME='CENTRAL', RPT_DIST_NO=182, PART_1_2=1, CRM_CD=330, CRM_CD_DESC='BURGLARY FROM VEHICLE', MOCODES='1822 1402 0344', VICT_AGE=47, VICT_SEX='M', PREMIS_CD=128, PREMIS_DESC='BUS STOP/LAYOVER (ALSO QUERY 124)', WEAPON_DESC=None, STATUS_DESC='INVEST CONT', CRM_CD_1=330, CRM_CD_2=998, CRM_CD_3=0, CRM_CD_4=0, LOCATION='1000 S  FLOWER                       ST', CROSS_STREET

In [5]:
# Convert data and timestamp columns to date type
cols_date_fix = ["DATE_OCC", "DATE_RPTD"]
for col in cols_date_fix:
    crime = crime.withColumn(col, F.trim(F.split(crime[col], " ").getItem(0)))
    crime = crime.withColumn(col, F.to_date(crime[col], "MM/dd/yyyy"))

In [6]:
# Remove extra spacing between LOCATION string
crime = crime.withColumn(
    "LOCATION",
    F.regexp_replace(F.col("LOCATION"), r"\s+", " ")
)

In [ ]:
# Show new table & schema
crime.show(10)
crime.printSchema()

# Save reformatted data as single file
crime.coalesce(1).write.csv("./data/data-reformat-pyspark", 
                header=True)

+---------+----------+----------+--------+----+----------+-----------+--------+------+--------------------+-------------------+--------+--------+---------+--------------------+-----------+------------+--------+--------+--------+--------+-------------------+------------+-------+---------+
|    DR_NO| DATE_RPTD|  DATE_OCC|TIME_OCC|AREA| AREA_NAME|RPT_DIST_NO|PART_1_2|CRM_CD|         CRM_CD_DESC|            MOCODES|VICT_AGE|VICT_SEX|PREMIS_CD|         PREMIS_DESC|WEAPON_DESC| STATUS_DESC|CRM_CD_1|CRM_CD_2|CRM_CD_3|CRM_CD_4|           LOCATION|CROSS_STREET|    LAT|      LON|
+---------+----------+----------+--------+----+----------+-----------+--------+------+--------------------+-------------------+--------+--------+---------+--------------------+-----------+------------+--------+--------+--------+--------+-------------------+------------+-------+---------+
|190326475|2020-03-01|2020-03-01|    2130|   7|  WILSHIRE|        784|       1|   510|    VEHICLE - STOLEN|               NULL|      

In [24]:
# Function to save dataframe to extracts folder
dest = "./extracts/pyspark/"

def save_df(df, fn):
    global dest
    df.coalesce(1).write.csv(dest + fn, header=True)

# Dictionary for pyspark queries
extracts = {}

# What are the most frequently reported crime types (CRM_CD_DESC) overall?

In [25]:
windowSpec = W.orderBy(F.col("TOTAL").desc())

freq_crimes = crime \
    .select(crime.CRM_CD,crime.CRM_CD_DESC) \
    .groupBy(crime.CRM_CD, crime.CRM_CD_DESC) \
    .agg(F.count("*").alias("TOTAL")) \
    .withColumn("RANK", 
                F.dense_rank().over(windowSpec)) \
    .where(F.col("RANK") == 1)
    
freq_crimes.show()
extracts.update({"frequent_crimes": freq_crimes})

25/10/24 00:15:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+------+----------------+------+----+
|CRM_CD|     CRM_CD_DESC| TOTAL|RANK|
+------+----------------+------+----+
|   510|VEHICLE - STOLEN|115230|   1|
+------+----------------+------+----+



25/10/24 00:15:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


# How many crimes were reported each month? Show a trend over time.

In [26]:
windowSpec = W.orderBy(F.col("YEAR"), F.col("MONTH"))

crime_trend = crime.select(
    crime.DATE_RPTD,
    F.year(crime.DATE_RPTD).alias("YEAR"),
    F.month(col=crime.DATE_RPTD).alias("MONTH")
) \
    .groupBy(F.col("YEAR"), F.col("MONTH")) \
    .agg(F.count("*").alias("COUNT")) \
    .withColumn("TREND_MM2MM",
                F.zeroifnull((F.lag(F.col("COUNT")).over(windowSpec)
                - F.col("COUNT"))* -1)) \
    .withColumn("TREND_RUNNING", 
                F.zeroifnull(F.sum(F.col("TREND_MM2MM")).over(windowSpec)))

crime_trend.show()
extracts.update({"crime_trend": crime_trend})

25/10/24 00:15:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----+-----+-----+-----------+-------------+
|YEAR|MONTH|COUNT|TREND_MM2MM|TREND_RUNNING|
+----+-----+-----+-----------+-------------+
|2020|    1|16060|          0|            0|
|2020|    2|16490|        430|          430|
|2020|    3|15539|       -951|         -521|
|2020|    4|15179|       -360|         -881|
|2020|    5|16331|       1152|          271|
|2020|    6|16955|        624|          895|
|2020|    7|16907|        -48|          847|
|2020|    8|16605|       -302|          545|
|2020|    9|15636|       -969|         -424|
|2020|   10|16142|        506|           82|
|2020|   11|15342|       -800|         -718|
|2020|   12|15522|        180|         -538|
|2021|    1|16155|        633|           95|
|2021|    2|15411|       -744|         -649|
|2021|    3|16307|        896|          247|
|2021|    4|16015|       -292|          -45|
|2021|    5|16784|        769|          724|
|2021|    6|17125|        341|         1065|
|2021|    7|18741|       1616|         2681|
|2021|    

25/10/24 00:15:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 0

# Which day of the week has the highest crime occurrences?

In [27]:
# Case statement converting weekday numeral to plain text.
case_day_num2word = """
CASE DAY_NUM
    WHEN 0 THEN 'Monday'
    WHEN 1 THEN 'Tuesday'
    WHEN 2 THEN 'Wednesday'
    WHEN 3 THEN 'Thursday'
    WHEN 4 THEN 'Friday'
    WHEN 5 THEN 'Saturday'
    WHEN 6 THEN 'Sunday'
END 
"""

day_most_crime = crime.select(
    F.weekday(crime.DATE_OCC).alias("DAY_NUM")
    ) \
    .withColumn("DAY_NAME", F.expr(case_day_num2word)) \
    .groupBy(F.col("DAY_NUM"), F.col("DAY_NAME")) \
    .agg(F.count("*").alias("TOTAL")) \
    .orderBy(F.col("TOTAL"))
    
day_most_crime.show()
extracts.update({"crime_weekday_high": day_most_crime})

+-------+---------+------+
|DAY_NUM| DAY_NAME| TOTAL|
+-------+---------+------+
|      1|  Tuesday|138151|
|      6|   Sunday|139652|
|      0|   Monday|141557|
|      3| Thursday|141841|
|      2|Wednesday|142733|
|      5| Saturday|147467|
|      4|   Friday|153703|
+-------+---------+------+



# What is the average number of crimes per area (AREA_NAME) per month?


In [28]:
crime_avg_area_mnth = crime \
    .select(
        F.month(crime.DATE_OCC).alias("MONTH"),
        F.year(crime.DATE_OCC).alias("YEAR"),
        crime.AREA_NAME
    ) \
    .groupBy("MONTH", "YEAR", crime.AREA_NAME) \
    .agg(F.count("*").alias("COUNT")) \
    .groupBy("AREA_NAME", "MONTH") \
    .agg(F.ceil(F.avg("COUNT")).alias("AVERAGE_CRIMES")) \
    .orderBy("AREA_NAME", "MONTH") 

crime_avg_area_mnth.show()
extracts.update({"crime_avg_area_mnth": crime_avg_area_mnth})


+-----------+-----+--------------+
|  AREA_NAME|MONTH|AVERAGE_CRIMES|
+-----------+-----+--------------+
|77TH STREET|    1|           931|
|77TH STREET|    2|           884|
|77TH STREET|    3|          1099|
|77TH STREET|    4|          1111|
|77TH STREET|    5|          1048|
|77TH STREET|    6|          1005|
|77TH STREET|    7|          1037|
|77TH STREET|    8|          1034|
|77TH STREET|    9|           985|
|77TH STREET|   10|          1014|
|77TH STREET|   11|           940|
|77TH STREET|   12|           906|
|    CENTRAL|    1|          1070|
|    CENTRAL|    2|           976|
|    CENTRAL|    3|          1115|
|    CENTRAL|    4|          1049|
|    CENTRAL|    5|          1112|
|    CENTRAL|    6|          1119|
|    CENTRAL|    7|          1213|
|    CENTRAL|    8|          1254|
+-----------+-----+--------------+
only showing top 20 rows


# How has crime volume changed year over year?


In [29]:
windowSpec = W.orderBy("YEAR")
crime_trend_year =  crime.select(crime.DATE_OCC) \
     .groupBy(F.year(crime.DATE_OCC).alias("YEAR")) \
     .agg(F.count("*").alias("COUNT"),
          F.count_distinct(F.month(crime.DATE_OCC)).alias("MONTHS_RPTD")) \
     .withColumn("TREND", F.col("COUNT") - F.lag("COUNT").over(windowSpec))

crime_trend_year.show()
extracts.update({"crime_trend_year": crime_trend_year})

25/10/24 00:15:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----+------+-----------+-------+
|YEAR| COUNT|MONTHS_RPTD|  TREND|
+----+------+-----------+-------+
|2020|199846|         12|   NULL|
|2021|209872|         12|  10026|
|2022|235256|         12|  25384|
|2023|232345|         12|  -2911|
|2024|127565|         12|-104780|
|2025|   220|          3|-127345|
+----+------+-----------+-------+



25/10/24 00:15:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


 # Which AREA_NAME reports the highest number of violent crimes (e.g., crimes with weapons)?

In [30]:
bad_crime_codes = ( 900, 930, 625, 940, 810, 113, 236, 910,
                    231, 926, 648, 121, 110, 250, 922, 230,
                    820, 760, 220, 812, 623, 822, 434, 763,
                    920, 840, 624, 815, 806, 627, 436, 942,
                    830, 761, 122, 901, 647, 821, 622, 626,
                    437, 921, 753, 435, 210, 251, 235, 860,
                    755 )

windowSpec = W.orderBy(F.col("BAD_CRIMES_CNT").desc())
crime_area_violent =    crime.filter(f"CRM_CD in {bad_crime_codes}") \
    .groupBy(crime.AREA_NAME) \
    .agg(F.count(crime.AREA_NAME).alias("BAD_CRIMES_CNT")) \
    .withColumn("RANK", 
                F.dense_rank().over(windowSpec)) \
    .filter("RANK = 1")

crime_area_violent.show()
extracts.update({"crime_area_violent": crime_area_violent})

25/10/24 00:15:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-----------+--------------+----+
|  AREA_NAME|BAD_CRIMES_CNT|RANK|
+-----------+--------------+----+
|77TH STREET|         28551|   1|
+-----------+--------------+----+



25/10/24 00:15:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


# What is the most common type of crime in each reporting district (RPT_DIST_NO)?

In [31]:
windowSpec = W.partitionBy("AREA_NAME").orderBy(F.col("COUNT").desc())

crime_common_area = crime.groupBy(crime.AREA_NAME, crime.CRM_CD_DESC) \
    .agg(F.count("*").alias("COUNT")) \
    .withColumn("RANK",
                F.rank().over(windowSpec)) \
    .filter("RANK = 1") \
    .orderBy("AREA_NAME") \

crime_common_area.show()
extracts.update({"crime_common_area": crime_common_area})

+-----------+--------------------+-----+----+
|  AREA_NAME|         CRM_CD_DESC|COUNT|RANK|
+-----------+--------------------+-----+----+
|77TH STREET|    VEHICLE - STOLEN| 8768|   1|
|    CENTRAL|BURGLARY FROM VEH...| 9695|   1|
| DEVONSHIRE|    VEHICLE - STOLEN| 3959|   1|
|   FOOTHILL|    VEHICLE - STOLEN| 4541|   1|
|     HARBOR|    VEHICLE - STOLEN| 6144|   1|
| HOLLENBECK|    VEHICLE - STOLEN| 6213|   1|
|  HOLLYWOOD|BATTERY - SIMPLE ...| 4482|   1|
|    MISSION|    VEHICLE - STOLEN| 6050|   1|
|N HOLLYWOOD|    VEHICLE - STOLEN| 5106|   1|
|     NEWTON|    VEHICLE - STOLEN| 8281|   1|
|  NORTHEAST|    VEHICLE - STOLEN| 5348|   1|
|    OLYMPIC|    VEHICLE - STOLEN| 5731|   1|
|    PACIFIC|    VEHICLE - STOLEN| 6527|   1|
|    RAMPART|    VEHICLE - STOLEN| 5441|   1|
|  SOUTHEAST|    VEHICLE - STOLEN| 7242|   1|
|  SOUTHWEST|    VEHICLE - STOLEN| 6678|   1|
|    TOPANGA|    VEHICLE - STOLEN| 3457|   1|
|   VAN NUYS|    VEHICLE - STOLEN| 4683|   1|
|    WEST LA|            BURGLARY|

# What are the top 5 locations (LOCATION) with the highest crime occurrences?

In [32]:
windowSpec = W.orderBy(F.col("COUNT").desc())

crime_high_location = crime.groupBy(crime.LOCATION) \
    .agg(F.count("*").alias("COUNT")) \
    .withColumn("RANK" ,
                F.rank().over(windowSpec)) \
    .filter("RANK <= 5")

crime_high_location.show()
extracts.update({"crime_high_location": crime_high_location})

25/10/24 00:15:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 0

+--------------------+-----+----+
|            LOCATION|COUNT|RANK|
+--------------------+-----+----+
|    800 N ALAMEDA ST| 2598|   1|
|   700 S FIGUEROA ST| 1711|   2|
|    100 THE GROVE DR| 1688|   3|
|10200 SANTA MONIC...| 1657|   4|
|              6TH ST| 1585|   5|
+--------------------+-----+----+



# Are there any significant crime clusters by latitude/longitude (LAT, LON)?

In [33]:
crime_clusters = crime.filter((F.col("LAT").isNotNull()) 
                              & (F.col("LON").isNotNull())) \
    .groupBy(F.round(crime.LAT, 2).alias("LAT_1KM_GRP"), 
             F.round(crime.LON, 2).alias("LON_1KM_GRP"), 
             crime.AREA_NAME) \
    .agg(F.count(crime.AREA_NAME).alias("TOTAL_EVENTS")) \
    .orderBy(F.col("TOTAL_EVENTS").desc()) 

crime_clusters.show(10)
extracts.update({"crime_clusters": crime_clusters})

+-----------+-----------+---------+------------+
|LAT_1KM_GRP|LON_1KM_GRP|AREA_NAME|TOTAL_EVENTS|
+-----------+-----------+---------+------------+
|      34.05|    -118.25|  CENTRAL|       13046|
|      34.05|    -118.26|  CENTRAL|       11214|
|       34.1|    -118.33|HOLLYWOOD|        8010|
|      34.04|    -118.25|  CENTRAL|        7883|
|      34.04|    -118.26|  CENTRAL|        7553|
|      34.06|    -118.27|  RAMPART|        6907|
|       34.1|    -118.34|HOLLYWOOD|        6897|
|      34.06|    -118.24|  CENTRAL|        5961|
|      34.05|    -118.24|  CENTRAL|        5855|
|      34.06|     -118.3|  OLYMPIC|        5699|
+-----------+-----------+---------+------------+
only showing top 10 rows


# At what times of day (TIME_OCC) are crimes most commonly reported?

In [34]:
daytime = {
    "NIGHT": (0, 559),
    "MORNING": (600, 1159),
    "NOON": (1200, 1759),
    "EVENING": (1800, 2359)
}

crime_common_time = crime.withColumn(
    "DAY_PART",
    F.when(F.col("TIME_OCC").between(*daytime["NIGHT"]), "NIGHT")
    .when(F.col("TIME_OCC").between(*daytime["MORNING"]), "MORNING")
    .when(F.col("TIME_OCC").between(*daytime["NOON"]), "NOON")
    .when(F.col("TIME_OCC").between(*daytime["EVENING"]), "EVENING")
    ) \
    .groupBy("DAY_PART") \
    .agg(F.count("*").alias("COUNT")) \
    .orderBy(F.col("COUNT").desc())

crime_common_time.show()
extracts.update({"crime_common_time": crime_common_time})

+--------+------+
|DAY_PART| COUNT|
+--------+------+
|    NOON|327355|
| EVENING|314122|
| MORNING|209929|
|   NIGHT|153698|
+--------+------+



# Are there specific hours where crime spike?

In [35]:

crime_spike_hours = crime.withColumn("DAY_HR",
                                     F.when(F.col("TIME_OCC") > 59, F.floor(F.col("TIME_OCC")/ 100))
                                     .otherwise(0)) \
    .groupBy("DAY_HR") \
    .agg(F.count("DAY_HR").alias("COUNT")) \
    .withColumn("RANK",
                F.dense_rank().over(W.orderBy(F.col("COUNT").desc()))) 

crime_spike_hours.show()
extracts.update({"crime_spike_hours": crime_spike_hours})


25/10/24 00:15:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+------+-----+----+
|DAY_HR|COUNT|RANK|
+------+-----+----+
|    12|67820|   1|
|    18|59964|   2|
|    17|58822|   3|
|    20|56354|   4|
|    19|55611|   5|
|    16|52983|   6|
|    15|52831|   7|
|    21|50796|   8|
|    14|49318|   9|
|    22|49115|  10|
|    13|45581|  11|
|    11|43668|  12|
|    10|43025|  13|
|    23|42282|  14|
|     0|40477|  15|
|     8|37250|  16|
|     9|36530|  17|
|     1|29763|  18|
|     7|26270|  19|
|     2|25217|  20|
+------+-----+----+
only showing top 20 rows


25/10/24 00:15:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/24 00:15:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [ ]:
# Access dictionary, save dataframes as csv locally
for name, output in extracts.items():
    print(name)
    save_df(df=output, fn=name)

frequent_crimes
crime_trend
crime_weekday_high
crime_avg_area_mnth
crime_trend_year
crime_area_violent
crime_common_area
crime_high_location
crime_clusters
crime_common_time
crime_spike_hours
